# Polynomiale Regression – Hypothesenräume vergleichen

In dieser Übung wählen Sie selbst verschiedene **Polynomgrade** und vergleichen, welches Modell auf bisher nicht verwendeten Testdaten am besten generalisiert.

**Ziel:**
- ein numerisches Feature `X` und eine numerische Zielvariable `Y` auswählen
- verschiedene Polynomgrade testen, z. B. 1, 2, 3, 5, 8
- für jedes Modell den **RMSE** auf Testdaten berechnen
- die gelernten Funktionen visualisieren
- Overfitting und Underfitting diskutieren


## 1. Datensatz hochladen

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)

# Häufige automatisch erzeugte Indexspalte entfernen
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

print(f'Datensatz: {filename}')
print(f'Zeilen: {df.shape[0]} | Spalten: {df.shape[1]}')
display(df.head())

## 2. Numerische Variablen anzeigen

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns.tolist()

print('Numerische Variablen:')
for col in numeric_columns:
    print('-', col)

## 3. Feature und Zielvariable auswählen

Ersetzen Sie die Beispielnamen durch zwei numerische Spalten Ihres Datensatzes.

In [ ]:
# Beispiel für Fußballer-Datensatz:
X_VARIABLE = 'age'
Y_VARIABLE = 'marketvalue'

assert X_VARIABLE in df.columns, f"Feature '{X_VARIABLE}' nicht gefunden."
assert Y_VARIABLE in df.columns, f"Zielvariable '{Y_VARIABLE}' nicht gefunden."

model_df = df[[X_VARIABLE, Y_VARIABLE]].dropna().copy()

print('Feature X:', X_VARIABLE)
print('Zielvariable Y:', Y_VARIABLE)
print('Verwendete Beobachtungen:', len(model_df))

## 4. Zusammenhang zunächst als Scatterplot betrachten

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(model_df[X_VARIABLE], model_df[Y_VARIABLE], alpha=0.5)
plt.xlabel(X_VARIABLE)
plt.ylabel(Y_VARIABLE)
plt.title(f'{Y_VARIABLE} vs. {X_VARIABLE}')
plt.grid(alpha=0.2)
plt.show()

## 5. Trainings- und Testdaten erzeugen

Das Modell wird auf den Trainingsdaten gelernt. Der RMSE wird auf **nicht zum Training verwendeten Testdaten** berechnet.

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df[[X_VARIABLE]]
y = model_df[Y_VARIABLE]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training:', len(X_train))
print('Test:', len(X_test))

## 6. Polynomgrade selbst festlegen

Wählen Sie die Hypothesenräume, die Sie vergleichen möchten.

Beispiele:
- Grad 1 → linear
- Grad 2 → quadratisch
- Grad 3 → kubisch
- höhere Grade → flexiblere Funktionen, aber höheres Overfitting-Risiko

In [ ]:
# Diese Liste können Sie verändern
DEGREES = [1, 2, 3, 5, 8]

print('Zu testende Polynomgrade:', DEGREES)

## 7. Modelle trainieren und RMSE vergleichen

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

results = []
models = {}

for degree in DEGREES:
    poly = PolynomialFeatures(degree=degree, include_bias=False)

    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    y_pred_train = model.predict(X_train_poly)
    y_pred_test = model.predict(X_test_poly)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    results.append({
        'Polynomgrad': degree,
        'RMSE Training': rmse_train,
        'RMSE Test': rmse_test
    })

    models[degree] = (poly, model)

results_df = pd.DataFrame(results).sort_values('RMSE Test')
display(results_df.round(2))

## 8. Bestes Modell bestimmen

Das Modell mit dem kleinsten **Test-RMSE** generalisiert von den getesteten Modellen am besten.

In [ ]:
best_row = results_df.iloc[0]
best_degree = int(best_row['Polynomgrad'])

print(f'Bester getesteter Polynomgrad: {best_degree}')
print(f"Test-RMSE: {best_row['RMSE Test']:.2f}")

## 9. Gelernte Funktionen visualisieren

Die Punkte sind die beobachteten Daten. Die Kurven sind die aus den Trainingsdaten gelernten Hypothesen.

In [ ]:
x_min = model_df[X_VARIABLE].min()
x_max = model_df[X_VARIABLE].max()
x_grid = np.linspace(x_min, x_max, 400).reshape(-1, 1)

plt.figure(figsize=(10,6))
plt.scatter(model_df[X_VARIABLE], model_df[Y_VARIABLE], alpha=0.25, label='Daten')

for degree in DEGREES:
    poly, model = models[degree]
    y_grid = model.predict(poly.transform(x_grid))
    plt.plot(x_grid[:,0], y_grid, label=f'Grad {degree}')

plt.xlabel(X_VARIABLE)
plt.ylabel(Y_VARIABLE)
plt.title('Vergleich verschiedener polynomialer Hypothesen')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 10. RMSE der Modelle visualisieren

In [ ]:
plot_results = pd.DataFrame(results).sort_values('Polynomgrad')

plt.figure(figsize=(8,5))
plt.plot(plot_results['Polynomgrad'], plot_results['RMSE Training'], marker='o', label='Training')
plt.plot(plot_results['Polynomgrad'], plot_results['RMSE Test'], marker='o', label='Test')
plt.xlabel('Polynomgrad')
plt.ylabel('RMSE')
plt.title('Modellkomplexität und Vorhersagefehler')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 11. Leitfragen

1. Welcher Polynomgrad hat den kleinsten Test-RMSE?
2. Welches Modell erscheint im Scatterplot zu einfach (**Underfitting**)?
3. Welches Modell folgt den Trainingsdaten möglicherweise zu stark (**Overfitting**)?
4. Wie unterscheiden sich Trainings-RMSE und Test-RMSE mit zunehmendem Polynomgrad?
5. Würden Sie auf Basis des Test-RMSE immer den höchsten getesteten Grad wählen? Warum nicht?
6. Welchen Hypothesenraum würden Sie für diesen Zusammenhang begründen?